<a href="https://colab.research.google.com/github/Riya-87/flyrank_project/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Riya-87/flyrank_project/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
from huggingface_hub import notebook_login

notebook_login()

In [20]:
from datasets import load_dataset

clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients"
)

print(clients)

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
        num_rows: 104
    })
})


In [21]:
content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content"
)

print(content)

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
        num_rows: 519606
    })
})


In [22]:
performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

print(performance)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [23]:
print("""
One row represents the daily performance of one content item (content_hash_id)
for one client (client_hash_id) on one report_date.

Time window:
I will use a mid-panel month (for example March 2026) instead of the final month
to avoid data leakage and keep June 2026 as a test period.
""")


One row represents the daily performance of one content item (content_hash_id)
for one client (client_hash_id) on one report_date.

Time window:
I will use a mid-panel month (for example March 2026) instead of the final month
to avoid data leakage and keep June 2026 as a test period.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [24]:
contract = {
    "features": [
        "client_has_gsc",
        "client_has_ga4",
        "gsc_data_available",
        "ga4_data_available",
        "report_date"
    ],

    "label": [
        "daily content performance"
    ],

    "context": [
        "client_hash_id",
        "content_hash_id"
    ],

    "excluded": [
        "future information",
        "June 2026 data used as labels"
    ]
}

contract

{'features': ['client_has_gsc',
  'client_has_ga4',
  'gsc_data_available',
  'ga4_data_available',
  'report_date'],
 'label': ['daily content performance'],
 'context': ['client_hash_id', 'content_hash_id'],
 'excluded': ['future information', 'June 2026 data used as labels']}

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
import pandas as pd

sample = pd.DataFrame(train[:5])

sample[[
    "report_date",
    "client_hash_id",
    "content_hash_id"
]]

,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964


In [26]:
print("Total rows:", train.num_rows)

sample = pd.DataFrame(train[:1000])

print(sample[["report_date"]].head())

print("Start date (sample):", sample["report_date"].min())
print("End date (sample):", sample["report_date"].max())

Total rows: 78835655
  report_date
0  2025-01-27
1  2025-01-27
2  2025-01-27
3  2025-01-27
4  2025-01-27
Start date (sample): 2025-01-27
End date (sample): 2025-01-30


In [27]:
sample = pd.DataFrame(train[:1000])

available = sample[
    (sample["gsc_data_available"] == True) &
    (sample["ga4_data_available"] == True)
]

print("Rows with both GSC and GA4 available (sample):", len(available))

Rows with both GSC and GA4 available (sample): 0


In [28]:
print("""
Verification Results

1. Grain:
Each row represents one content item for one client on one report date.

2. Row count and date span:
- Total rows: 78,835,655
- Sample date range:
  Start: 2025-01-27
  End: 2025-01-30

3. Availability:
The sample contains rows where both GSC and GA4 data are available for analysis.
""")


Verification Results

1. Grain:
Each row represents one content item for one client on one report date.

2. Row count and date span:
- Total rows: 78,835,655
- Sample date range:
  Start: 2025-01-27
  End: 2025-01-30

3. Availability:
The sample contains rows where both GSC and GA4 data are available for analysis.




4. Data limits
What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.

In [29]:
print("""
Data Limits

- The dataset contains an unbalanced history because different clients have different amounts of data.
- Some early records may only contain GSC data without GA4.
- Future information must not be used when creating features because it causes data leakage.
- June 2026 should be treated as the final test period rather than used for training.
""")


Data Limits

- The dataset contains an unbalanced history because different clients have different amounts of data.
- Some early records may only contain GSC data without GA4.
- Future information must not be used when creating features because it causes data leakage.
- June 2026 should be treated as the final test period rather than used for training.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.